# Загрузка готовых данных ATG (Amazon Toys & Games)

Файлы получены от авторов AlphaFuse/iDreamRec. Цель ноутбука:

1. Понять структуру каждого файла.
2. Проверить совместимость с заявленной статистикой (AlphaFuse Table 1: 19,412 users / 11,924 items / 138,444 interactions).
3. Проверить размерность эмбеддингов и распределение взаимодействий.
4. Сделать sanity checks (нет ли утечки между сплитами, всем ли товарам есть эмбеддинги).
5. Подготовить унифицированные структуры данных для следующих шагов (whitening, dataloader, FM).

## Положи файлы в `./data/atg/`:
- `3large_emb.pickle` — эмбеддинги от OpenAI text-embedding-3-large (ожидаемая размерность 3072).
- `data.df` — основной dataframe.
- `train_data.df`, `val_data.df`, `test_data.df` — сплиты.

In [2]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100
pd.set_option('display.max_colwidth', 80)

DATA_DIR = Path('../data/ATG')
OUT_DIR = Path('../data/processed')
OUT_DIR.mkdir(parents=True, exist_ok=True)

files = {
    'emb':   DATA_DIR / '3large_emb.pickle',
    'data':  DATA_DIR / 'data.df',
    'train': DATA_DIR / 'train_data.df',
    'val':   DATA_DIR / 'val_data.df',
    'test':  DATA_DIR / 'test_data.df',
}

for name, path in files.items():
    if path.exists():
        size_mb = path.stat().st_size / 1024 / 1024
        print(f'  [OK]   {name:6s}  {path}  ({size_mb:.1f} MB)')
    else:
        print(f'  [MISS] {name:6s}  {path}')

  [OK]   emb     ..\data\ATG\3large_emb.pickle  (279.5 MB)
  [MISS] data    ..\data\ATG\data.df
  [OK]   train   ..\data\ATG\train_data.df  (0.7 MB)
  [OK]   val     ..\data\ATG\val_data.df  (0.1 MB)
  [OK]   test    ..\data\ATG\test_data.df  (0.1 MB)


## 1. Эмбеддинги

Не знаем заранее формат — может быть `dict[item_id -> np.ndarray]`, может быть `(matrix, list_of_ids)`, может быть готовый `pd.DataFrame`. Универсальный подход: загружаем через pickle и разбираем по факту.

In [3]:
with open(files['emb'], 'rb') as f:
    emb_obj = pickle.load(f)

print(f'Type: {type(emb_obj).__name__}')

# Разбираем основные варианты
if isinstance(emb_obj, dict):
    print(f'Dict with {len(emb_obj):,} keys')
    first_key = next(iter(emb_obj))
    first_val = emb_obj[first_key]
    print(f'  First key:   {first_key!r}  (type: {type(first_key).__name__})')
    print(f'  First value: type={type(first_val).__name__}')
    if hasattr(first_val, "shape"):
        print(f'               shape={first_val.shape}, dtype={first_val.dtype}')
elif isinstance(emb_obj, tuple):
    print(f'Tuple of length {len(emb_obj)}')
    for i, x in enumerate(emb_obj):
        info = f'type={type(x).__name__}'
        if hasattr(x, 'shape'):
            info += f', shape={x.shape}'
        elif hasattr(x, '__len__'):
            info += f', len={len(x)}'
        print(f'  [{i}] {info}')
elif isinstance(emb_obj, np.ndarray):
    print(f'ndarray: shape={emb_obj.shape}, dtype={emb_obj.dtype}')
elif isinstance(emb_obj, pd.DataFrame):
    print(f'DataFrame: shape={emb_obj.shape}')
    print('Columns:', list(emb_obj.columns))
    print(emb_obj.head())
else:
    print('Unknown structure, raw:')
    print(emb_obj)

Type: ndarray
ndarray: shape=(11924, 3072), dtype=float64


In [6]:
# Нормализация в единый вид: emb_matrix (N, D), item_ids (N,)
# Подставь нужную ветку в зависимости от того, что выше показала диагностика

emb_matrix = emb_obj

print(f'emb_matrix: shape={emb_matrix.shape}, dtype={emb_matrix.dtype}')
print(f'item_ids:   N={len(item_ids)}, type of id: {type(item_ids[0]).__name__}')
print(f'Sample ids: {item_ids[:5]}')
print(f'Sample embedding (first 10 dims): {emb_matrix[0, :10]}')

emb_matrix: shape=(11924, 3072), dtype=float64


NameError: name 'item_ids' is not defined

In [7]:
# Базовая статистика по эмбеддингам
print('Embedding stats per dimension:')
print(f'  mean of means:  {emb_matrix.mean(axis=0).mean():+.4f}')
print(f'  mean of stds:   {emb_matrix.std(axis=0).mean():.4f}')
print(f'  global min:     {emb_matrix.min():+.4f}')
print(f'  global max:     {emb_matrix.max():+.4f}')

# OpenAI обычно возвращает L2-нормализованные векторы. Проверим
norms = np.linalg.norm(emb_matrix, axis=1)
print(f'\nL2 norms: mean={norms.mean():.4f}, std={norms.std():.4f}, min={norms.min():.4f}, max={norms.max():.4f}')
# Если mean ~1.0 и std маленький — эмбеддинги нормализованы на сферу.

Embedding stats per dimension:
  mean of means:  +0.0001
  mean of stds:   0.0150
  global min:     -0.1648
  global max:     +0.1758

L2 norms: mean=1.0000, std=0.0000, min=1.0000, max=1.0000


## 2. Сплиты

Те же неизвестные: какие колонки, что в строках, как представлены последовательности.

In [ ]:
def load_df(path):
    """Файл с расширением .df — это pickle с DataFrame внутри."""
    return pd.read_pickle(path)

data_df = load_df(files['data'])
print(f'data.df: shape={data_df.shape}')
print('Columns:', list(data_df.columns))
print('Dtypes:')
print(data_df.dtypes)
print('\nHead:')
data_df.head()

In [ ]:
train_df = load_df(files['train'])
val_df   = load_df(files['val'])
test_df  = load_df(files['test'])

for name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    print(f'{name}: shape={df.shape}, columns={list(df.columns)}')

print('\ntrain_df head:')
train_df.head()

In [ ]:
# Посмотрим в детали одну строку каждого split
for name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    print(f'\n=== {name} (first row) ===')
    row = df.iloc[0]
    for col in df.columns:
        val = row[col]
        val_repr = str(val)
        if len(val_repr) > 200:
            val_repr = val_repr[:200] + '...'
        print(f'  {col}: {val_repr}')

## 3. Анализ последовательностей

Какова длина истории? Сколько уникальных юзеров и айтемов? Совпадает ли это со статистикой AlphaFuse Table 1?

**Важно**: я не знаю точно, в каких колонках лежат `user_id`, `seq`, `target`. Скорректируй имена колонок в коде ниже по результатам диагностики выше.

In [ ]:
# === ЗАМЕНИ имена колонок на фактические из диагностики выше ===
USER_COL   = 'user_id'      # колонка с user id
SEQ_COL    = 'seq'          # колонка с историей (список item ids)
TARGET_COL = 'next'         # колонка с таргет item (тот, что надо предсказать)

# Если структура другая, например target лежит как последний элемент seq, поправь логику

def seq_stats(df, name):
    if SEQ_COL not in df.columns:
        print(f'{name}: column {SEQ_COL!r} not found')
        return
    lens = df[SEQ_COL].apply(len)
    print(f'{name}: rows={len(df):,}, seq_len: min={lens.min()}, '
          f'mean={lens.mean():.2f}, median={lens.median():.0f}, max={lens.max()}')

for name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    seq_stats(df, name)

In [ ]:
# Сводная статистика — для сверки с AlphaFuse Table 1:
# Toys: 19,412 users / 11,924 items / 138,444 interactions

all_users = set()
all_items = set()
total_interactions = 0

for df in [train_df, val_df, test_df]:
    if USER_COL in df.columns:
        all_users.update(df[USER_COL].unique())
    if SEQ_COL in df.columns:
        for seq in df[SEQ_COL]:
            all_items.update(seq)
            total_interactions += len(seq)
    if TARGET_COL in df.columns:
        all_items.update(df[TARGET_COL].unique())
        total_interactions += len(df)

print(f'Total unique users:        {len(all_users):,}')
print(f'Total unique items:        {len(all_items):,}')
print(f'Total interactions:        {total_interactions:,}')
print()
print(f'AlphaFuse Table 1 (Toys):  19,412 users / 11,924 items / 138,444 interactions')

## 4. Sanity checks

Перед тем как переходить к модели, проверим, что данные согласованы между собой.

In [ ]:
# 4.1 Все ли item ids в данных имеют эмбеддинги?
emb_ids_set = set(item_ids)
missing = all_items - emb_ids_set
extra = emb_ids_set - all_items

print(f'Items in data, no embedding:    {len(missing):,}')
print(f'Items with embedding, not used: {len(extra):,}')
if missing:
    sample = list(missing)[:5]
    print(f'  Sample missing: {sample}')
if extra and len(extra) < 100:
    sample = list(extra)[:5]
    print(f'  Sample extra:   {sample}')

In [ ]:
# 4.2 Утечка пользователей между сплитами (cold-start setting должен исключать пересечения)
if USER_COL in train_df.columns:
    train_users = set(train_df[USER_COL])
    val_users   = set(val_df[USER_COL])
    test_users  = set(test_df[USER_COL])
    print(f'Train users: {len(train_users):,}')
    print(f'Val users:   {len(val_users):,}')
    print(f'Test users:  {len(test_users):,}')
    print(f'\nIntersection train ∩ val:  {len(train_users & val_users):,}')
    print(f'Intersection train ∩ test: {len(train_users & test_users):,}')
    print(f'Intersection val ∩ test:   {len(val_users & test_users):,}')
    print('\nЕсли пересечения == 0 — это cold-start split. Если != 0 — leave-one-out.')

In [ ]:
# 4.3 Распределение длин последовательностей
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5), sharey=True)
for ax, (name, df) in zip(axes, [('train', train_df), ('val', val_df), ('test', test_df)]):
    if SEQ_COL not in df.columns:
        continue
    lens = df[SEQ_COL].apply(len)
    ax.hist(lens, bins=range(int(lens.min()), int(lens.max()) + 2), edgecolor='black', alpha=0.7)
    ax.set_title(f'{name}: seq lengths (mean={lens.mean():.2f})')
    ax.set_xlabel('length')
axes[0].set_ylabel('count')
plt.tight_layout()
plt.show()

In [ ]:
# 4.4 Покажем один пример полной последовательности
row = test_df.iloc[0]
print(f'User: {row.get(USER_COL, "?")}')
print(f'Sequence length: {len(row[SEQ_COL])}')
print(f'History items: {row[SEQ_COL]}')
if TARGET_COL in test_df.columns:
    print(f'Target item:   {row[TARGET_COL]}')

## 5. Семантика эмбеддингов — sanity check

Если эмбеддинги хорошие, ближайшие соседи в этом пространстве должны быть тематически похожими товарами. Без знания текстов проверить сложно, но можем хотя бы посмотреть на распределение косинусных сходств — раз эмбеддинги от OpenAI, они анизотропны и среднее сходство будет высоким.

In [ ]:
# Нормализуем (если ещё не нормализованы), считаем косинусы для случайной выборки пар
rng = np.random.default_rng(42)
sample_idx = rng.choice(len(emb_matrix), size=min(5000, len(emb_matrix)), replace=False)
sample = emb_matrix[sample_idx]
sample_normed = sample / np.linalg.norm(sample, axis=1, keepdims=True)

# Случайные пары — 10000 штук
i = rng.integers(0, len(sample), size=10000)
j = rng.integers(0, len(sample), size=10000)
mask = i != j
cos_sims = (sample_normed[i[mask]] * sample_normed[j[mask]]).sum(axis=1)

print(f'Cosine similarity of random item pairs:')
print(f'  mean:   {cos_sims.mean():.4f}')
print(f'  median: {np.median(cos_sims):.4f}')
print(f'  std:    {cos_sims.std():.4f}')
print(f'  min:    {cos_sims.min():.4f}')
print(f'  max:    {cos_sims.max():.4f}')

fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(cos_sims, bins=50, edgecolor='black', alpha=0.7)
ax.set_xlabel('cosine similarity')
ax.set_ylabel('count')
ax.set_title(f'Pairwise cosine similarity — {len(cos_sims):,} random pairs')
plt.tight_layout()
plt.show()

# Если mean ≈ 0.3-0.6 — это типичная анизотропия. Whitening её исправит.

## 6. Сохранение в унифицированном виде

Чтобы дальше в моделинг-пайплайне не возиться с pickle-форматами и неизвестными колонками, сохраним всё в чистом виде:

- `embeddings.npz` — матрица + порядок item_ids (item_id → index лукап).
- `train.parquet`, `val.parquet`, `test.parquet` — со стандартными колонками `user_id`, `history` (list of item indices), `target` (item index).

Индексы вместо string id — это гораздо быстрее в DataLoader.

In [ ]:
# Создаём маппинг item_id -> int
item_id_to_idx = {item_id: i for i, item_id in enumerate(item_ids)}

# Сохраняем эмбеддинги: матрица + список ids в том же порядке
np.savez(
    OUT_DIR / 'embeddings.npz',
    emb=emb_matrix,
    item_ids=np.array(item_ids, dtype=object),
)
print(f'Saved embeddings: {OUT_DIR / "embeddings.npz"}')
print(f'  Shape: {emb_matrix.shape}, dtype: {emb_matrix.dtype}')

In [ ]:
def to_indices(seq):
    """Список item_id -> список индексов. Неизвестные пропускаем (не должно быть, но на всякий)."""
    return [item_id_to_idx[x] for x in seq if x in item_id_to_idx]

def standardize_df(df, name):
    out = pd.DataFrame()
    if USER_COL in df.columns:
        out['user_id'] = df[USER_COL].values
    if SEQ_COL in df.columns:
        out['history'] = df[SEQ_COL].apply(to_indices).values
    if TARGET_COL in df.columns:
        out['target'] = df[TARGET_COL].map(item_id_to_idx).values
    out.to_parquet(OUT_DIR / f'{name}.parquet', index=False)
    print(f'Saved {name}.parquet: {len(out):,} rows, columns={list(out.columns)}')
    return out

for name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    standardize_df(df, name)

In [ ]:
# Финальная сверка: всё лежит, всё читается
print('\nFinal contents:')
for p in sorted(OUT_DIR.iterdir()):
    size_mb = p.stat().st_size / 1024 / 1024
    print(f'  {p.name}  ({size_mb:.2f} MB)')

# Тест: можем ли мы быстро поднять данные обратно
print('\nRead-back test:')
loaded = np.load(OUT_DIR / 'embeddings.npz', allow_pickle=True)
print(f'  emb shape: {loaded["emb"].shape}')
print(f'  n_items:   {len(loaded["item_ids"])}')

tr = pd.read_parquet(OUT_DIR / 'train.parquet')
print(f'  train: {len(tr):,} rows')
print(f'  sample row: history len = {len(tr.iloc[0]["history"])}, target = {tr.iloc[0]["target"]}')

## Что дальше

1. **Whitening эмбеддингов** — считаем μ и Σ на train items, применяем ко всем. Это нужно для flow matching из N(0, I).
2. **PyTorch Dataset/DataLoader** — обёртка над parquet + npz, выдающая батчи `(history_emb, target_emb)`.
3. **Архитектура модели** — Transformer encoder для истории + MLP denoiser + FM loss.
4. **Train loop** — с EMA, logging, валидацией HR@K/NDCG@K на val.